Cell 1：彻底清理旧 PRESENT + 重新 clone

In [13]:
# ============================================================
# Cell 1
# Clean old PRESENT workspace and clone official repository
# ============================================================

from pathlib import Path
import shutil
import subprocess
import sys


WORK_ROOT = Path(
    "/kaggle/working"
)

PRESENT_ROOT = (
    WORK_ROOT
    / "PRESENT"
)

PRESENT_OUTPUT_ROOT = (
    WORK_ROOT
    / "PRESENT_baseline"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)


print("=" * 100)
print("RESET PRESENT WORKSPACE")
print("=" * 100)


# ------------------------------------------------------------
# Remove old PRESENT repository
# ------------------------------------------------------------

if PRESENT_ROOT.exists():

    shutil.rmtree(
        PRESENT_ROOT
    )

    print(
        "Deleted old repo:",
        PRESENT_ROOT
    )


# ------------------------------------------------------------
# Remove old failed PRESENT outputs
# ------------------------------------------------------------

if PRESENT_OUTPUT_ROOT.exists():

    shutil.rmtree(
        PRESENT_OUTPUT_ROOT
    )

    print(
        "Deleted old outputs:",
        PRESENT_OUTPUT_ROOT
    )


# ------------------------------------------------------------
# Clear stale modules if this kernel was previously used
# ------------------------------------------------------------

stale_modules = [

    name

    for name in list(sys.modules)

    if (
        name == "PRESENT"
        or name.startswith("PRESENT.")
        or name == "episcanpy"
        or name.startswith("episcanpy.")
    )
]


for name in stale_modules:

    del sys.modules[name]


print(
    "Cleared stale modules:",
    len(stale_modules)
)


# ------------------------------------------------------------
# Clone official repository
# ------------------------------------------------------------

subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/lizhen18THU/PRESENT.git",
        str(PRESENT_ROOT),
    ],
    check=True,
)


assert PRESENT_ROOT.exists()

assert DATA_ROOT.exists()


# ------------------------------------------------------------
# Record exact commit
# ------------------------------------------------------------

PRESENT_COMMIT = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(PRESENT_ROOT),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)


print(
    "\nPRESENT root:",
    PRESENT_ROOT
)

print(
    "Data root   :",
    DATA_ROOT
)

print(
    "Git commit  :",
    PRESENT_COMMIT
)


print(
    "\nPASS: clean PRESENT workspace ready."
)

RESET PRESENT WORKSPACE
Deleted old repo: /kaggle/working/PRESENT
Deleted old outputs: /kaggle/working/PRESENT_baseline
Cleared stale modules: 63


Cloning into '/kaggle/working/PRESENT'...



PRESENT root: /kaggle/working/PRESENT
Data root   : /kaggle/input/datasets/wuvdji/smgc-data
Git commit  : c88a609b34aae9b84c2c8a7ffb6824bae5c78f23

PASS: clean PRESENT workspace ready.


Cell 2：安装正确依赖

In [14]:
# ============================================================
# Cell 2
# Install PRESENT runtime for Kaggle Python 3.12
#
# Important:
#   episcanpy == 0.3.2
#
# Keep current:
#   Python 3.12
#   PyTorch 2.10
#   CUDA 12.8
# ============================================================

import sys
import subprocess
import torch


print("=" * 100)
print("INSTALL PRESENT DEPENDENCIES")
print("=" * 100)

print(
    "Current torch:",
    torch.__version__
)

print(
    "CUDA:",
    torch.version.cuda
)


# ------------------------------------------------------------
# Remove any incompatible epiScanpy installation first
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "episcanpy",
    ],
    check=False,
)


# ------------------------------------------------------------
# General dependencies
# ------------------------------------------------------------

packages = [

    "anndata==0.11.4",
    "scanpy==1.11.4",
    "scikit-misc",

    "leidenalg",
    "python-igraph",
    "louvain",

    "genomicranges",
    "iranges",
    "biocutils",

    "torch-geometric==2.8.0.post1",
]


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *packages,
    ],
    check=True,
)


# ------------------------------------------------------------
# Official PRESENT epiScanpy version
#
# --no-deps prevents old epiScanpy requirements from
# downgrading the entire Kaggle environment.
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "episcanpy==0.3.2",
    ],
    check=True,
)


# ------------------------------------------------------------
# PyG binary extensions for torch 2.10 + CUDA 12.8
# ------------------------------------------------------------

PYG_WHEEL_URL = (
    "https://data.pyg.org/whl/"
    "torch-2.10.0+cu128.html"
)


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",

        "pyg_lib",
        "torch_scatter",
        "torch_sparse",
        "torch_cluster",

        "-f",
        PYG_WHEEL_URL,
    ],
    check=True,
)


# ------------------------------------------------------------
# Install official PRESENT source without dependency downgrade
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(PRESENT_ROOT),
        "--no-deps",
    ],
    check=True,
)


print(
    "\nPASS: PRESENT dependencies installed."
)

INSTALL PRESENT DEPENDENCIES
Current torch: 2.10.0+cu128
CUDA: 12.8
Found existing installation: episcanpy 0.3.2
Uninstalling episcanpy-0.3.2:
  Successfully uninstalled episcanpy-0.3.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 25.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 19.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 66.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 28.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bio-present 1.0.2 requires episcanpy==0.3.2, which is not installed.
bio-present 1.0.2 requires anndata==0.9.2, but you have anndata 0.11.4 which is incompatible.
bio-present 1.0.2 requires biocutils==0.1.3, but you have biocutils 0.5.0 which is incompatible.
bio-present 1.0.2 requires genomicranges==0.4.2, but you have genomicranges 0.9.0 which is incompatible.
bio-present 1.0.2 requires iranges==0.2.1, but you have iranges 0.7.3 which is incompatible.
bio-present 1.0.2 requires leidenalg==0.9.1, but you have leidenalg 0.12.0 which is incompatible.
bio-present 1.0.2 requires louvain==0.8.0, but you have louvain 0.7.1 which is incompatible.
bio-present 1.0.2 requires networkx==3.1, but you have networkx 3.6.1 which is incompatible.
bio-present 1.0.2 requires numpy==1.24.4, but you have numpy 2.0.2 which is incompa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 53.7 MB/s eta 0:00:00

PASS: PRESENT dependencies installed.


Cell 3：兼容补丁 + 环境完整检查

In [15]:
# ============================================================
# Cell 3
# PRESENT compatibility + environment audit
#
# Compatibility only:
#   NumPy 2.x legacy aliases
#   SciPy sparse .A
#
# No PRESENT source modification.
# ============================================================

import sys
import importlib


# ============================================================
# 1. NumPy compatibility
# ============================================================

import numpy as np


if not hasattr(np, "NaN"):
    np.NaN = np.nan

if not hasattr(np, "Inf"):
    np.Inf = np.inf

if not hasattr(np, "Infinity"):
    np.Infinity = np.inf

if not hasattr(np, "infty"):
    np.infty = np.inf


# ============================================================
# 2. SciPy compatibility
# ============================================================

import scipy
import scipy.sparse as sp


if not hasattr(
    sp.spmatrix,
    "A",
):

    sp.spmatrix.A = property(
        lambda self:
            self.toarray()
    )


for cls in [
    sp.csr_matrix,
    sp.csc_matrix,
    sp.coo_matrix,
]:

    if not hasattr(
        cls,
        "A",
    ):

        cls.A = property(
            lambda self:
                self.toarray()
        )


# ============================================================
# 3. Official repository first in sys.path
# ============================================================

repo_path = str(
    PRESENT_ROOT
)


sys.path = [

    p

    for p in sys.path

    if p != repo_path
]


sys.path.insert(
    0,
    repo_path,
)


importlib.invalidate_caches()


# ============================================================
# 4. Imports
# ============================================================

import pandas as pd
import sklearn
import torch
import scanpy as sc
import anndata
import torch_geometric

import episcanpy.api as epi


from PRESENT.Main import (
    PRESENT_function,
)

from PRESENT.Utils import (
    setup_seed,
    run_leiden,
)


import pyg_lib
import torch_sparse
import torch_cluster
import torch_scatter


# ============================================================
# 5. Verify epiScanpy
# ============================================================

from importlib.metadata import version


EPISCANPY_VERSION = version(
    "episcanpy"
)


assert (
    EPISCANPY_VERSION
    == "0.3.2"
)


# ============================================================
# 6. Environment report
# ============================================================

print("=" * 100)
print("PRESENT ENVIRONMENT AUDIT")
print("=" * 100)

print(
    "Python        :",
    sys.version.split()[0]
)

print(
    "PyTorch       :",
    torch.__version__
)

print(
    "CUDA runtime  :",
    torch.version.cuda
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU           :",
        torch.cuda.get_device_name(0)
    )


print(
    "NumPy         :",
    np.__version__
)

print(
    "SciPy         :",
    scipy.__version__
)

print(
    "Scanpy        :",
    sc.__version__
)

print(
    "AnnData       :",
    anndata.__version__
)

print(
    "sklearn       :",
    sklearn.__version__
)

print(
    "PyG           :",
    torch_geometric.__version__
)

print(
    "epiScanpy     :",
    EPISCANPY_VERSION
)


assert torch.cuda.is_available()

assert hasattr(
    np,
    "NaN"
)

assert hasattr(
    sp.csr_matrix([[1]]),
    "A"
)


print(
    "\nPASS: epiScanpy 0.3.2 active."
)

print(
    "PASS: PRESENT import successful."
)

print(
    "PASS: NumPy compatibility active."
)

print(
    "PASS: SciPy compatibility active."
)

print(
    "PASS: PRESENT environment ready."
)

PRESENT ENVIRONMENT AUDIT
Python        : 3.12.13
PyTorch       : 2.10.0+cu128
CUDA runtime  : 12.8
CUDA available: True
GPU           : Tesla T4
NumPy         : 2.0.2
SciPy         : 1.16.3
Scanpy        : 1.11.4
AnnData       : 0.11.4
sklearn       : 1.6.1
PyG           : 2.8.0.post1
epiScanpy     : 0.3.2

PASS: epiScanpy 0.3.2 active.
PASS: PRESENT import successful.
PASS: NumPy compatibility active.
PASS: SciPy compatibility active.
PASS: PRESENT environment ready.


Cell 4：5 个数据集 loader + 固定 spot compatibility

In [16]:
# ============================================================
# Cell 4
# Dataset loader + frozen benchmark population compatibility
# ============================================================

from contextlib import contextmanager

import numpy as np
import scanpy as sc
import scipy.sparse as sp
import episcanpy.api as epi


# ============================================================
# Dataset definitions
# ============================================================

DATASET_SPECS = {

    "HLN-A1": {

        "rna":
            DATA_ROOT
            / "Human_Lymph_Nodes/A1/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Human_Lymph_Nodes/A1/adata_ADT.h5ad",

        "second_type":
            "ADT",

        "K":
            10,

        "n_spots":
            3484,
    },


    "HLN-D1": {

        "rna":
            DATA_ROOT
            / "Human_Lymph_Nodes/D1/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Human_Lymph_Nodes/D1/adata_ADT.h5ad",

        "second_type":
            "ADT",

        "K":
            11,

        "n_spots":
            3359,
    },


    "E18.5": {

        "rna":
            DATA_ROOT
            / "E18.5_mouse_brain/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "E18.5_mouse_brain/adata_ATAC.h5ad",

        "second_type":
            "ATAC",

        "K":
            14,

        "n_spots":
            2129,
    },


    "S2-E15": {

        "rna":
            DATA_ROOT
            / "Mouse_Embryos_S2/E15/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Mouse_Embryos_S2/E15/adata_ATAC.h5ad",

        "second_type":
            "ATAC",

        "K":
            15,

        "n_spots":
            1939,
    },


    "S2-E18": {

        "rna":
            DATA_ROOT
            / "Mouse_Embryos_S2/E18/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Mouse_Embryos_S2/E18/adata_ATAC.h5ad",

        "second_type":
            "ATAC",

        "K":
            16,

        "n_spots":
            2248,
    },
}


# ============================================================
# Load raw data
# ============================================================

def load_present_dataset(
    dataset_name,
):

    assert (
        dataset_name
        in DATASET_SPECS
    )


    spec = DATASET_SPECS[
        dataset_name
    ]


    assert spec["rna"].exists(), (
        spec["rna"]
    )

    assert spec["second"].exists(), (
        spec["second"]
    )


    rna = sc.read_h5ad(
        spec["rna"]
    )

    second = sc.read_h5ad(
        spec["second"]
    )


    # Remove duplicate-feature-name warnings downstream.
    rna.var_names_make_unique()
    second.var_names_make_unique()


    assert (
        rna.n_obs
        == spec["n_spots"]
    )

    assert (
        second.n_obs
        == spec["n_spots"]
    )


    assert np.array_equal(
        np.asarray(
            rna.obs_names
        ),
        np.asarray(
            second.obs_names
        ),
    )


    # ========================================================
    # E18.5 metadata adapter
    # ========================================================

    if dataset_name == "E18.5":

        coords = np.column_stack(
            [
                np.asarray(
                    rna.obs[
                        "array_col"
                    ],
                    dtype=np.float32,
                ),

                np.asarray(
                    rna.obs[
                        "array_row"
                    ],
                    dtype=np.float32,
                ),
            ]
        )


        labels = (
            rna.obs[
                "Combined_Clusters_annotation"
            ]
            .astype(str)
            .to_numpy()
        )


        assert (
            len(
                np.unique(labels)
            )
            == 14
        )


        for adata in [
            rna,
            second,
        ]:

            adata.obsm[
                "spatial"
            ] = coords.copy()

            adata.obs[
                "Spatial_Label"
            ] = labels.copy()


    # ========================================================
    # Common audit
    # ========================================================

    for adata in [
        rna,
        second,
    ]:

        assert (
            "Spatial_Label"
            in adata.obs.columns
        )

        assert (
            "spatial"
            in adata.obsm
        )


        assert (
            np.asarray(
                adata.obsm[
                    "spatial"
                ]
            ).shape
            == (
                spec["n_spots"],
                2,
            )
        )


    assert np.allclose(
        np.asarray(
            rna.obsm["spatial"]
        ),
        np.asarray(
            second.obsm["spatial"]
        ),
    )


    assert (
        rna.obs[
            "Spatial_Label"
        ].nunique()
        == spec["K"]
    )


    print(
        f"{dataset_name}: "
        f"{spec['n_spots']} spots | "
        f"K={spec['K']} | "
        f"RNA + {spec['second_type']}"
    )


    return (
        rna,
        second,
        spec,
    )


# ============================================================
# Frozen-population compatibility
#
# Disable ONLY observation-level filtering during
# PRESENT_function().
#
# Retain:
#   sc.pp.filter_genes()
#   epi.pp.filter_features()
#   HVG
#   model
#   graph
#   early stopping
#   Leiden
# ============================================================

@contextmanager
def present_keep_all_spots():

    original_sc_filter_cells = (
        sc.pp.filter_cells
    )

    original_epi_filter_cells = (
        epi.pp.filter_cells
    )


    def _keep_sc_cells(
        data,
        *args,
        **kwargs,
    ):

        # Deliberate no-op:
        # benchmark observations are frozen.
        return None


    def _keep_epi_cells(
        data,
        *args,
        **kwargs,
    ):

        # Deliberate no-op:
        # benchmark observations are frozen.
        return None


    sc.pp.filter_cells = (
        _keep_sc_cells
    )

    epi.pp.filter_cells = (
        _keep_epi_cells
    )


    try:

        yield

    finally:

        sc.pp.filter_cells = (
            original_sc_filter_cells
        )

        epi.pp.filter_cells = (
            original_epi_filter_cells
        )


print(
    "\nPASS: dataset loader defined."
)

print(
    "PASS: RNA/ADT/ATAC observation filtering compatibility defined."
)


PASS: dataset loader defined.
PASS: RNA/ADT/ATAC observation filtering compatibility defined.


Cell 5：定义通用 PRESENT 单次 runner

In [17]:
# ============================================================
# Cell 5
# FINAL reusable PRESENT single-run runner
# ============================================================

from pathlib import Path
import json
import gc
import time

import numpy as np
import torch

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


PRESENT_OUTPUT_ROOT = Path(
    "/kaggle/working/PRESENT_baseline"
)


def dataset_folder_name(
    dataset_name,
):

    return {

        "HLN-A1": "HLNA1",
        "HLN-D1": "HLND1",
        "E18.5": "E185",
        "S2-E15": "S2E15",
        "S2-E18": "S2E18",

    }[
        dataset_name
    ]


def run_present_once(
    dataset_name,
    seed,
    stage="smoke",
):

    # ========================================================
    # 1. Fresh raw dataset for EVERY run
    # ========================================================

    rna, second, spec = (
        load_present_dataset(
            dataset_name
        )
    )


    original_spot_ids = np.asarray(
        rna.obs_names.astype(str)
    )


    assert (
        len(
            np.unique(
                original_spot_ids
            )
        )
        == spec["n_spots"]
    )


    # ========================================================
    # 2. Seed
    # ========================================================

    setup_seed(
        seed
    )


    # ========================================================
    # 3. Output directory
    # ========================================================

    out_dir = (
        PRESENT_OUTPUT_ROOT
        / stage
        / (
            dataset_folder_name(
                dataset_name
            )
            + f"_seed{seed}"
        )
    )


    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    print(
        "\n" + "=" * 110
    )

    print(
        f"PRESENT | "
        f"{dataset_name} | "
        f"seed={seed}"
    )

    print(
        "=" * 110
    )


    print(
        "Input spots:",
        spec["n_spots"]
    )

    print(
        "Target K   :",
        spec["K"]
    )

    print(
        "Modality   :",
        f"RNA + {spec['second_type']}"
    )


    # ========================================================
    # 4. Official recommended parameters
    # ========================================================

    common_kwargs = {

        "spatial_key":
            "spatial",

        "batch_key":
            None,

        "adata_rna":
            rna,

        "gene_min_cells":
            1,

        "num_hvg":
            3000,

        "nclusters":
            spec["K"],

        "d_lat":
            50,

        "k_neighbors":
            6,

        "epochs":
            100,

        "lr":
            1e-3,

        "batch_size":
            320,

        "device":
            "cuda",

        "device_id":
            0,
    }


    start_time = time.time()


    # ========================================================
    # 5. Run official PRESENT
    #
    # Only observation-level filters are temporarily disabled.
    # ========================================================

    with present_keep_all_spots():

        if (
            spec["second_type"]
            == "ADT"
        ):

            result = PRESENT_function(

                **common_kwargs,

                adata_adt=second,

                protein_min_cells=1,
            )


        elif (
            spec["second_type"]
            == "ATAC"
        ):

            result = PRESENT_function(

                **common_kwargs,

                adata_atac=second,

                peak_min_cells_fraction=0.03,
            )


        else:

            raise ValueError(
                spec[
                    "second_type"
                ]
            )


    elapsed = (
        time.time()
        - start_time
    )


    # ========================================================
    # 6. Frozen-population audit
    # ========================================================

    print(
        "\nOutput spots:",
        result.n_obs
    )


    assert (
        result.n_obs
        == spec["n_spots"]
    ), (
        f"{dataset_name}: population changed "
        f"{spec['n_spots']} -> "
        f"{result.n_obs}"
    )


    result_spot_ids = np.asarray(
        result.obs_names.astype(str)
    )


    assert (
        set(result_spot_ids)
        == set(original_spot_ids)
    ), (
        f"{dataset_name}: output spot set changed."
    )


    assert (
        len(result_spot_ids)
        == len(
            np.unique(
                result_spot_ids
            )
        )
    )


    # ========================================================
    # 7. Extract output
    # ========================================================

    assert (
        "embeddings"
        in result.obsm
    )

    assert (
        "LeidenClusters"
        in result.obs.columns
    )

    assert (
        "Spatial_Label"
        in result.obs.columns
    )


    embedding = np.asarray(
        result.obsm[
            "embeddings"
        ]
    )


    gt = (
        result.obs[
            "Spatial_Label"
        ]
        .astype(str)
        .to_numpy()
    )


    pred = (
        result.obs[
            "LeidenClusters"
        ]
        .astype(str)
        .to_numpy()
    )


    coords = np.asarray(
        result.obsm[
            "spatial"
        ]
    )


    n_pred = len(
        np.unique(pred)
    )


    # ========================================================
    # 8. Structural assertions
    # ========================================================

    assert (
        embedding.shape
        == (
            spec["n_spots"],
            50,
        )
    ), (
        f"Unexpected embedding shape: "
        f"{embedding.shape}"
    )


    assert np.isfinite(
        embedding
    ).all()


    assert (
        len(gt)
        == spec["n_spots"]
    )


    assert (
        len(
            np.unique(gt)
        )
        == spec["K"]
    )


    assert (
        n_pred
        == spec["K"]
    ), (
        f"Expected K={spec['K']}, "
        f"got {n_pred}"
    )


    assert (
        coords.shape
        == (
            spec["n_spots"],
            2,
        )
    )


    # ========================================================
    # 9. Unified evaluation
    # ========================================================

    ari = adjusted_rand_score(
        gt,
        pred,
    )


    nmi = normalized_mutual_info_score(
        gt,
        pred,
        average_method="max",
    )


    # ========================================================
    # 10. Report
    # ========================================================

    print(
        "\n" + "=" * 110
    )

    print(
        f"{dataset_name} PRESENT RESULT"
    )

    print(
        "=" * 110
    )


    print(
        "Embedding shape:",
        embedding.shape
    )

    print(
        "Predicted K    :",
        n_pred
    )

    print(
        f"ARI = {ari:.12f}"
    )

    print(
        f"NMI = {nmi:.12f}"
    )

    print(
        f"Runtime = {elapsed:.2f}s"
    )


    # ========================================================
    # 11. Save evidence
    # ========================================================

    np.save(
        out_dir
        / "embedding.npy",
        embedding,
    )

    np.save(
        out_dir
        / "pred_labels.npy",
        pred,
    )

    np.save(
        out_dir
        / "gt_labels.npy",
        gt,
    )

    np.save(
        out_dir
        / "coords.npy",
        coords,
    )

    np.save(
        out_dir
        / "spot_ids.npy",
        result_spot_ids,
    )


    metrics = {

        "method":
            "PRESENT",

        "dataset":
            dataset_name,

        "training_seed":
            int(seed),

        "benchmark_population":
            "frozen_all_spots",

        "n_spots":
            int(
                result.n_obs
            ),

        "n_clusters":
            int(
                spec["K"]
            ),

        "predicted_clusters":
            int(
                n_pred
            ),

        "modalities":
            (
                "RNA+"
                + spec[
                    "second_type"
                ]
            ),

        "gene_min_cells":
            1,

        "num_hvg":
            3000,

        "protein_min_cells":
            (
                1
                if
                spec["second_type"]
                == "ADT"
                else None
            ),

        "peak_min_cells_fraction":
            (
                0.03
                if
                spec["second_type"]
                == "ATAC"
                else None
            ),

        "d_lat":
            50,

        "k_neighbors":
            6,

        "max_epochs":
            100,

        "lr":
            0.001,

        "batch_size":
            320,

        "clustering":
            "PRESENT_official_Leiden",

        "ARI":
            float(
                ari
            ),

        "NMI":
            float(
                nmi
            ),

        "NMI_average_method":
            "max",

        "runtime_seconds":
            float(
                elapsed
            ),

        "PRESENT_git_commit":
            PRESENT_COMMIT,

        "compatibility":
            {
                "numpy_legacy_alias":
                    True,

                "scipy_sparse_A":
                    True,

                "disable_scanpy_observation_filtering":
                    True,

                "disable_episcanpy_observation_filtering":
                    True,
            },
    }


    with (
        out_dir
        / "metrics.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            metrics,
            f,
            indent=2,
        )


    print(
        "\nSaved:",
        out_dir
    )

    print(
        "\nPASS: PRESENT run completed."
    )

    print(
        f"PASS: {spec['n_spots']}/"
        f"{spec['n_spots']} spots evaluated."
    )


    # ========================================================
    # 12. Cleanup
    # ========================================================

    del rna
    del second
    del result

    gc.collect()

    torch.cuda.empty_cache()


    return metrics

Cell 6：HLN-A1 seed0 smoke

In [18]:
# ============================================================
# Cell 6
# PRESENT HLN-A1 seed0 smoke
# ============================================================

HLNA1_SMOKE = run_present_once(
    dataset_name="HLN-A1",
    seed=0,
    stage="smoke",
)


print(
    "\n" + "=" * 100
)

print(
    "HLN-A1 SMOKE COMPLETE"
)

print(
    "=" * 100
)


print(
    "ARI:",
    HLNA1_SMOKE[
        "ARI"
    ]
)

print(
    "NMI:",
    HLNA1_SMOKE[
        "NMI"
    ]
)

HLN-A1: 3484 spots | K=10 | RNA + ADT

PRESENT | HLN-A1 | seed=0
Input spots: 3484
Target K   : 10
Modality   : RNA + ADT
Loading data and parameters...
Input data has been loaded


Computing METIS partitioning...
Done!
Model training:  41%|████      | 41/100 [00:20<00:29,  1.98it/s, NLL_loss=0.373, BNN_loss=0.275, MSE_loss=0.354, IOA_loss=0.039, ES counter=20, ES patience=20] 


Early stop the training process


Succeed to find 10 clusters at resolution 0.891

Output spots: 3484

HLN-A1 PRESENT RESULT
Embedding shape: (3484, 50)
Predicted K    : 10
ARI = 0.230099373952
NMI = 0.329626002574
Runtime = 42.90s

Saved: /kaggle/working/PRESENT_baseline/smoke/HLNA1_seed0

PASS: PRESENT run completed.
PASS: 3484/3484 spots evaluated.

HLN-A1 SMOKE COMPLETE
ARI: 0.23009937395197963
NMI: 0.3296260025741663


Cell 7：E18.5 seed0 smoke

In [19]:
# ============================================================
# Cell 7
# PRESENT E18.5 seed0 smoke
# ============================================================

E185_SMOKE = run_present_once(
    dataset_name="E18.5",
    seed=0,
    stage="smoke",
)


print(
    "\n" + "=" * 100
)

print(
    "E18.5 SMOKE COMPLETE"
)

print(
    "=" * 100
)


print(
    "ARI:",
    E185_SMOKE[
        "ARI"
    ]
)

print(
    "NMI:",
    E185_SMOKE[
        "NMI"
    ]
)

E18.5: 2129 spots | K=14 | RNA + ATAC

PRESENT | E18.5 | seed=0
Input spots: 2129
Target K   : 14
Modality   : RNA + ATAC
Loading data and parameters...


Input data has been loaded


Computing METIS partitioning...
Done!
Model training:  44%|████▍     | 44/100 [03:39<04:39,  4.99s/it, NLL_loss=0.684, BNN_loss=0.251, MSE_loss=0.439, IOA_loss=0.12, ES counter=20, ES patience=20] 


Early stop the training process


Succeed to find 14 clusters at resolution 0.750

Output spots: 2129

E18.5 PRESENT RESULT
Embedding shape: (2129, 50)
Predicted K    : 14
ARI = 0.596911584065
NMI = 0.642572203591
Runtime = 241.72s

Saved: /kaggle/working/PRESENT_baseline/smoke/E185_seed0

PASS: PRESENT run completed.
PASS: 2129/2129 spots evaluated.

E18.5 SMOKE COMPLETE
ARI: 0.596911584064751
NMI: 0.6425722035906251


Cell 8：冻结 PRESENT 正式协议

In [20]:
# ============================================================
# Cell 8
# Freeze PRESENT formal benchmark protocol
# ============================================================

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
import sklearn
import scipy
import scanpy as sc
import anndata
import torch_geometric

from importlib.metadata import version


FORMAL_ROOT = Path(
    "/kaggle/working/PRESENT_baseline/formal_10seeds"
)

FORMAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


FORMAL_SEEDS = list(
    range(10)
)


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


EXPECTED = {

    "HLN-A1": {
        "n_spots": 3484,
        "K": 10,
        "modality": "RNA+ADT",
    },

    "HLN-D1": {
        "n_spots": 3359,
        "K": 11,
        "modality": "RNA+ADT",
    },

    "E18.5": {
        "n_spots": 2129,
        "K": 14,
        "modality": "RNA+ATAC",
    },

    "S2-E15": {
        "n_spots": 1939,
        "K": 15,
        "modality": "RNA+ATAC",
    },

    "S2-E18": {
        "n_spots": 2248,
        "K": 16,
        "modality": "RNA+ATAC",
    },
}


FORMAL_PROTOCOL = {

    "method":
        "PRESENT",

    "git_commit":
        PRESENT_COMMIT,

    "seeds":
        FORMAL_SEEDS,

    "datasets":
        EXPECTED,

    "benchmark_population":
        "frozen_all_spots",

    "observation_filtering":
        (
            "disabled to preserve identical "
            "benchmark spot populations"
        ),

    "feature_preprocessing":
        "official PRESENT",

    "gene_min_cells":
        1,

    "num_hvg":
        3000,

    "protein_min_cells":
        1,

    "peak_min_cells_fraction":
        0.03,

    "d_lat":
        50,

    "k_neighbors":
        6,

    "max_epochs":
        100,

    "lr":
        0.001,

    "batch_size":
        320,

    "clustering":
        "official PRESENT Leiden",

    "n_clusters":
        "ground-truth K",

    "ARI":
        "sklearn adjusted_rand_score",

    "NMI":
        'sklearn normalized_mutual_info_score(average_method="max")',

    "std_ddof":
        0,

    "compatibility": {

        "episcanpy":
            "0.3.2",

        "numpy_legacy_alias":
            True,

        "scipy_sparse_A":
            True,

        "source_algorithm_modified":
            False,
    },

    "software": {

        "python":
            sys.version.split()[0],

        "torch":
            torch.__version__,

        "numpy":
            np.__version__,

        "scipy":
            scipy.__version__,

        "sklearn":
            sklearn.__version__,

        "scanpy":
            sc.__version__,

        "anndata":
            anndata.__version__,

        "torch_geometric":
            torch_geometric.__version__,

        "episcanpy":
            version("episcanpy"),
    },
}


with (
    FORMAL_ROOT
    / "protocol.json"
).open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        FORMAL_PROTOCOL,
        f,
        indent=2,
    )


print("=" * 100)
print("PRESENT FORMAL PROTOCOL")
print("=" * 100)

print(
    "Output:",
    FORMAL_ROOT,
)

print(
    "Seeds:",
    FORMAL_SEEDS,
)

print(
    "Commit:",
    PRESENT_COMMIT,
)

print(
    "\nPASS: PRESENT formal protocol frozen."
)

PRESENT FORMAL PROTOCOL
Output: /kaggle/working/PRESENT_baseline/formal_10seeds
Seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Commit: c88a609b34aae9b84c2c8a7ffb6824bae5c78f23

PASS: PRESENT formal protocol frozen.


Cell 9：50-run 正式 runner，可断点续跑

In [21]:
# ============================================================
# Cell 9
# PRESENT formal benchmark
#
# 5 datasets x seeds 0..9
# Resume-safe:
#   valid metrics.json -> skip
# ============================================================

from pathlib import Path
import json
import shutil
import traceback
import gc

import numpy as np
import pandas as pd
import torch


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


FORMAL_SEEDS = list(
    range(10)
)


FORMAL_ROOT = Path(
    "/kaggle/working/PRESENT_baseline/formal_10seeds"
)


# ============================================================
# Helper
# ============================================================

def formal_run_dir(
    dataset,
    seed,
):

    return (
        PRESENT_OUTPUT_ROOT
        / "formal_10seeds"
        / (
            dataset_folder_name(dataset)
            + f"_seed{seed}"
        )
    )


def valid_existing_run(
    dataset,
    seed,
):

    run_dir = formal_run_dir(
        dataset,
        seed,
    )

    metrics_path = (
        run_dir
        / "metrics.json"
    )


    if not metrics_path.exists():

        return None


    try:

        with metrics_path.open(
            "r",
            encoding="utf-8",
        ) as f:

            m = json.load(f)


        expected = EXPECTED[
            dataset
        ]


        required = [
            "dataset",
            "training_seed",
            "n_spots",
            "predicted_clusters",
            "ARI",
            "NMI",
        ]


        if not all(
            key in m
            for key in required
        ):
            return None


        if (
            m["dataset"]
            != dataset
        ):
            return None


        if (
            int(
                m["training_seed"]
            )
            != seed
        ):
            return None


        if (
            int(
                m["n_spots"]
            )
            != expected["n_spots"]
        ):
            return None


        if (
            int(
                m["predicted_clusters"]
            )
            != expected["K"]
        ):
            return None


        # Essential evidence
        for name in [
            "embedding.npy",
            "pred_labels.npy",
            "gt_labels.npy",
            "coords.npy",
            "spot_ids.npy",
        ]:

            if not (
                run_dir
                / name
            ).exists():

                return None


        return m


    except Exception:

        return None


# ============================================================
# Main loop
# ============================================================

rows = []


for dataset in DATASET_ORDER:

    print(
        "\n\n" + "#" * 110
    )

    print(
        f"# FORMAL DATASET: {dataset}"
    )

    print(
        "#" * 110
    )


    for seed in FORMAL_SEEDS:

        existing = (
            valid_existing_run(
                dataset,
                seed,
            )
        )


        if existing is not None:

            print(
                f"[SKIP] "
                f"{dataset} seed={seed} | "
                f"ARI={existing['ARI']:.6f} | "
                f"NMI={existing['NMI']:.6f}"
            )

            rows.append(
                existing
            )

            continue


        print(
            "\n" + "-" * 100
        )

        print(
            f"RUN "
            f"{dataset} "
            f"seed={seed}"
        )

        print(
            "-" * 100
        )


        try:

            metrics = (
                run_present_once(
                    dataset_name=dataset,
                    seed=seed,
                    stage="formal_10seeds",
                )
            )


            # -----------------------------------------------
            # Formal run audit immediately after completion
            # -----------------------------------------------

            assert (
                metrics["n_spots"]
                == EXPECTED[
                    dataset
                ][
                    "n_spots"
                ]
            )


            assert (
                metrics[
                    "predicted_clusters"
                ]
                == EXPECTED[
                    dataset
                ][
                    "K"
                ]
            )


            assert np.isfinite(
                metrics["ARI"]
            )


            assert np.isfinite(
                metrics["NMI"]
            )


            rows.append(
                metrics
            )


        except Exception as e:

            run_dir = formal_run_dir(
                dataset,
                seed,
            )

            run_dir.mkdir(
                parents=True,
                exist_ok=True,
            )


            error_info = {

                "dataset":
                    dataset,

                "seed":
                    seed,

                "error":
                    repr(e),

                "traceback":
                    traceback.format_exc(),
            }


            with (
                run_dir
                / "ERROR.json"
            ).open(
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    error_info,
                    f,
                    indent=2,
                )


            print(
                "\nFAILED:"
            )

            print(
                repr(e)
            )


            gc.collect()

            torch.cuda.empty_cache()


# ============================================================
# Reconstruct results from DISK
#
# Do not trust only current Python memory.
# ============================================================

disk_rows = []


for dataset in DATASET_ORDER:

    for seed in FORMAL_SEEDS:

        m = valid_existing_run(
            dataset,
            seed,
        )


        if m is not None:

            disk_rows.append(
                m
            )


raw_df = pd.DataFrame(
    disk_rows
)


# ============================================================
# Always save current progress
# ============================================================

RAW_PATH = (
    FORMAL_ROOT
    / "PRESENT_5datasets_10seeds_RAW.csv"
)


raw_df.to_csv(
    RAW_PATH,
    index=False,
)


print(
    "\n" + "=" * 110
)

print(
    "CURRENT RUN COUNTS"
)

print(
    "=" * 110
)


if len(raw_df) > 0:

    counts = (
        raw_df
        .groupby(
            "dataset"
        )
        .size()
    )

    print(
        counts
    )

else:

    counts = pd.Series(
        dtype=int
    )

    print(
        "No successful runs."
    )


print(
    "\nTotal successful runs:",
    len(raw_df),
    "/ 50"
)


# ============================================================
# Only create final summary if all 50 passed
# ============================================================

if len(raw_df) == 50:

    for dataset in DATASET_ORDER:

        assert (
            int(
                counts[
                    dataset
                ]
            )
            == 10
        )


    summary_rows = []


    for dataset in DATASET_ORDER:

        part = (
            raw_df[
                raw_df[
                    "dataset"
                ]
                == dataset
            ]
            .sort_values(
                "training_seed"
            )
        )


        assert (
            sorted(
                part[
                    "training_seed"
                ].astype(int)
            )
            == list(
                range(10)
            )
        )


        summary_rows.append(
            {

                "dataset":
                    dataset,

                "n_runs":
                    10,

                "ARI_mean":
                    float(
                        part[
                            "ARI"
                        ].mean()
                    ),

                "ARI_std":
                    float(
                        part[
                            "ARI"
                        ].std(
                            ddof=0
                        )
                    ),

                "NMI_mean":
                    float(
                        part[
                            "NMI"
                        ].mean()
                    ),

                "NMI_std":
                    float(
                        part[
                            "NMI"
                        ].std(
                            ddof=0
                        )
                    ),
            }
        )


    summary_df = pd.DataFrame(
        summary_rows
    )


    SUMMARY_PATH = (
        FORMAL_ROOT
        / "PRESENT_5datasets_10seeds_SUMMARY.csv"
    )


    summary_df.to_csv(
        SUMMARY_PATH,
        index=False,
    )


    print(
        "\n" + "=" * 110
    )

    print(
        "PRESENT FORMAL SUMMARY"
    )

    print(
        "=" * 110
    )


    for _, row in (
        summary_df.iterrows()
    ):

        print(
            f"{row['dataset']:8s} | "
            f"ARI "
            f"{row['ARI_mean']:.6f}"
            f" ± "
            f"{row['ARI_std']:.6f}"
            f" | NMI "
            f"{row['NMI_mean']:.6f}"
            f" ± "
            f"{row['NMI_std']:.6f}"
        )


    print(
        "\nPASS: 50/50 PRESENT formal runs completed."
    )


else:

    print(
        "\nFormal experiment is incomplete."
    )

    print(
        "Re-run this SAME Cell 9 later."
    )

    print(
        "Completed runs will be skipped automatically."
    )



##############################################################################################################
# FORMAL DATASET: HLN-A1
##############################################################################################################

----------------------------------------------------------------------------------------------------
RUN HLN-A1 seed=0
----------------------------------------------------------------------------------------------------


HLN-A1: 3484 spots | K=10 | RNA + ADT

PRESENT | HLN-A1 | seed=0
Input spots: 3484
Target K   : 10
Modality   : RNA + ADT
Loading data and parameters...
Input data has been loaded


Computing METIS partitioning...
Done!
Model training:  41%|████      | 41/100 [00:20<00:29,  1.99it/s, NLL_loss=0.373, BNN_loss=0.275, MSE_loss=0.354, IOA_loss=0.039, ES counter=20, ES patience=20] 


Early stop the training process


Succeed to find 10 clusters at resolution 0.891

Output spots: 3484

HLN-A1 PRESENT RESULT
Embedding shape: (3484, 50)
Predicted K    : 10
ARI = 0.245447004856
NMI = 0.332531193917
Runtime = 40.49s

Saved: /kaggle/working/PRESENT_baseline/formal_10seeds/HLNA1_seed0

PASS: PRESENT run completed.
PASS: 3484/3484 spots evaluated.

----------------------------------------------------------------------------------------------------
RUN HLN-A1 seed=1
----------------------------------------------------------------------------------------------------


...

S2-E18: 2248 spots | K=16 | RNA + ATAC

PRESENT | S2-E18 | seed=9
Input spots: 2248
Target K   : 16
Modality   : RNA + ATAC
Loading data and parameters...


Input data has been loaded


Computing METIS partitioning...
Done!
Model training:  38%|███▊      | 38/100 [01:54<03:06,  3.01s/it, NLL_loss=0.954, BNN_loss=0.238, MSE_loss=0.463, IOA_loss=0.0915, ES counter=20, ES patience=20]


Early stop the training process


Succeed to find 16 clusters at resolution 1.312

Output spots: 2248

S2-E18 PRESENT RESULT
Embedding shape: (2248, 50)
Predicted K    : 16
ARI = 0.307729933072
NMI = 0.482412053896
Runtime = 128.40s

Saved: /kaggle/working/PRESENT_baseline/formal_10seeds/S2E18_seed9

PASS: PRESENT run completed.
PASS: 2248/2248 spots evaluated.

CURRENT RUN COUNTS
dataset
E18.5      9
HLN-A1    10
HLN-D1    10
S2-E15     9
S2-E18    10
dtype: int64

Total successful runs: 48 / 50

Formal experiment is incomplete.
Re-run this SAME Cell 9 later.
Completed runs will be skipped automatically.


新建 Cell 9.1：只补跑两个缺失 seed

In [24]:
# ============================================================
# Cell 9.1
# Recover the two PRESENT runs rejected only because
# official Leiden did not hit exactly the target K.
#
# Missing:
#   E18.5  seed=1 : target K=14, official fallback K=15
#   S2-E15 seed=8 : target K=15, official fallback K=14
#
# IMPORTANT:
# We accept PRESENT's official run_leiden() fallback behavior.
# We do NOT force/merge/split clusters.
# ============================================================

from pathlib import Path
import json
import gc
import time

import numpy as np
import torch

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


RECOVERY_RUNS = [
    ("E18.5", 1),
    ("S2-E15", 8),
]


def recover_present_official_fallback(
    dataset_name,
    seed,
):

    # ========================================================
    # 1. Load fresh raw data
    # ========================================================

    rna, second, spec = (
        load_present_dataset(
            dataset_name
        )
    )


    original_spot_ids = np.asarray(
        rna.obs_names.astype(str)
    )


    assert (
        len(original_spot_ids)
        == spec["n_spots"]
    )


    # ========================================================
    # 2. Seed
    # ========================================================

    setup_seed(
        seed
    )


    # ========================================================
    # 3. Formal output directory
    # ========================================================

    out_dir = (
        PRESENT_OUTPUT_ROOT
        / "formal_10seeds"
        / (
            dataset_folder_name(
                dataset_name
            )
            + f"_seed{seed}"
        )
    )


    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    print(
        "\n" + "=" * 110
    )

    print(
        f"RECOVERY | PRESENT | "
        f"{dataset_name} | seed={seed}"
    )

    print(
        "=" * 110
    )


    print(
        "Input spots:",
        spec["n_spots"]
    )

    print(
        "Target K   :",
        spec["K"]
    )


    # ========================================================
    # 4. Same frozen protocol
    # ========================================================

    common_kwargs = {

        "spatial_key":
            "spatial",

        "batch_key":
            None,

        "adata_rna":
            rna,

        "gene_min_cells":
            1,

        "num_hvg":
            3000,

        "nclusters":
            spec["K"],

        "d_lat":
            50,

        "k_neighbors":
            6,

        "epochs":
            100,

        "lr":
            1e-3,

        "batch_size":
            320,

        "device":
            "cuda",

        "device_id":
            0,
    }


    start_time = time.time()


    # ========================================================
    # 5. Official PRESENT
    # ========================================================

    with present_keep_all_spots():

        if (
            spec["second_type"]
            == "ADT"
        ):

            result = PRESENT_function(

                **common_kwargs,

                adata_adt=second,

                protein_min_cells=1,
            )


        elif (
            spec["second_type"]
            == "ATAC"
        ):

            result = PRESENT_function(

                **common_kwargs,

                adata_atac=second,

                peak_min_cells_fraction=0.03,
            )


        else:

            raise ValueError(
                spec["second_type"]
            )


    elapsed = (
        time.time()
        - start_time
    )


    # ========================================================
    # 6. Frozen population audit
    # ========================================================

    assert (
        result.n_obs
        == spec["n_spots"]
    )


    result_spot_ids = np.asarray(
        result.obs_names.astype(str)
    )


    assert (
        set(result_spot_ids)
        == set(original_spot_ids)
    )


    # ========================================================
    # 7. Extract official output
    # ========================================================

    embedding = np.asarray(
        result.obsm[
            "embeddings"
        ]
    )


    gt = (
        result.obs[
            "Spatial_Label"
        ]
        .astype(str)
        .to_numpy()
    )


    pred = (
        result.obs[
            "LeidenClusters"
        ]
        .astype(str)
        .to_numpy()
    )


    coords = np.asarray(
        result.obsm[
            "spatial"
        ]
    )


    actual_k = int(
        len(
            np.unique(pred)
        )
    )


    assert (
        embedding.shape
        == (
            spec["n_spots"],
            50,
        )
    )


    assert np.isfinite(
        embedding
    ).all()


    assert (
        len(
            np.unique(gt)
        )
        == spec["K"]
    )


    # IMPORTANT:
    # DO NOT assert actual_k == target K.
    #
    # PRESENT official run_leiden() itself returns
    # the last Leiden partition when exact K cannot
    # be found.
    assert (
        actual_k > 1
    )


    # ========================================================
    # 8. Independent metrics
    # ========================================================

    ari = adjusted_rand_score(
        gt,
        pred,
    )


    nmi = (
        normalized_mutual_info_score(
            gt,
            pred,
            average_method="max",
        )
    )


    exact_match = (
        actual_k
        == spec["K"]
    )


    print(
        "\nTarget K       :",
        spec["K"]
    )

    print(
        "Official output K:",
        actual_k
    )

    print(
        "Exact K match  :",
        exact_match
    )

    print(
        f"ARI = {ari:.12f}"
    )

    print(
        f"NMI = {nmi:.12f}"
    )


    if not exact_match:

        print(
            "\nNOTE:"
        )

        print(
            "Official PRESENT run_leiden() "
            "did not reach exact target K."
        )

        print(
            "Its returned fallback partition "
            "is retained unchanged."
        )


    # ========================================================
    # 9. Save evidence
    # ========================================================

    np.save(
        out_dir
        / "embedding.npy",
        embedding,
    )


    np.save(
        out_dir
        / "pred_labels.npy",
        pred,
    )


    np.save(
        out_dir
        / "gt_labels.npy",
        gt,
    )


    np.save(
        out_dir
        / "coords.npy",
        coords,
    )


    np.save(
        out_dir
        / "spot_ids.npy",
        result_spot_ids,
    )


    metrics = {

        "method":
            "PRESENT",

        "dataset":
            dataset_name,

        "training_seed":
            int(seed),

        "benchmark_population":
            "frozen_all_spots",

        "n_spots":
            int(
                result.n_obs
            ),

        # Requested benchmark target
        "n_clusters":
            int(
                spec["K"]
            ),

        # Actual official Leiden output
        "predicted_clusters":
            actual_k,

        "cluster_count_exact_match":
            bool(
                exact_match
            ),

        "clustering_fallback":
            (
                None
                if exact_match
                else
                "official_run_leiden_returned_last_partition"
            ),

        "modalities":
            (
                "RNA+"
                + spec[
                    "second_type"
                ]
            ),

        "gene_min_cells":
            1,

        "num_hvg":
            3000,

        "protein_min_cells":
            (
                1
                if
                spec["second_type"]
                == "ADT"
                else None
            ),

        "peak_min_cells_fraction":
            (
                0.03
                if
                spec["second_type"]
                == "ATAC"
                else None
            ),

        "d_lat":
            50,

        "k_neighbors":
            6,

        "max_epochs":
            100,

        "lr":
            0.001,

        "batch_size":
            320,

        "clustering":
            "PRESENT_official_Leiden",

        "ARI":
            float(ari),

        "NMI":
            float(nmi),

        "NMI_average_method":
            "max",

        "runtime_seconds":
            float(elapsed),

        "PRESENT_git_commit":
            PRESENT_COMMIT,

        "compatibility":
            {

                "numpy_legacy_alias":
                    True,

                "scipy_sparse_A":
                    True,

                "disable_scanpy_observation_filtering":
                    True,

                "disable_episcanpy_observation_filtering":
                    True,
            },
    }


    with (
        out_dir
        / "metrics.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            metrics,
            f,
            indent=2,
        )


    # Remove old failure record now that
    # the official output has been accepted.
    error_path = (
        out_dir
        / "ERROR.json"
    )


    if error_path.exists():

        error_path.unlink()


    print(
        "\nSaved:",
        out_dir
    )

    print(
        "PASS: recovery run saved."
    )


    del rna
    del second
    del result

    gc.collect()

    torch.cuda.empty_cache()


    return metrics


# ============================================================
# Run ONLY the two missing runs
# ============================================================

RECOVERED = []


for dataset_name, seed in RECOVERY_RUNS:

    recovered = (
        recover_present_official_fallback(
            dataset_name,
            seed,
        )
    )

    RECOVERED.append(
        recovered
    )


print(
    "\n" + "=" * 110
)

print(
    "RECOVERY COMPLETE"
)

print(
    "=" * 110
)


for m in RECOVERED:

    print(
        f"{m['dataset']:8s} "
        f"seed={m['training_seed']} | "
        f"target K={m['n_clusters']} | "
        f"actual K={m['predicted_clusters']} | "
        f"ARI={m['ARI']:.6f} | "
        f"NMI={m['NMI']:.6f}"
    )

E18.5: 2129 spots | K=14 | RNA + ATAC

RECOVERY | PRESENT | E18.5 | seed=1
Input spots: 2129
Target K   : 14
Loading data and parameters...


Input data has been loaded


Computing METIS partitioning...
Done!
Model training:  41%|████      | 41/100 [03:24<04:54,  4.99s/it, NLL_loss=0.686, BNN_loss=0.22, MSE_loss=0.441, IOA_loss=0.12, ES counter=20, ES patience=20]  


Early stop the training process


Succeed to find 14 clusters at resolution 0.891

Target K       : 14
Official output K: 14
Exact K match  : True
ARI = 0.489642286806
NMI = 0.585966121798

Saved: /kaggle/working/PRESENT_baseline/formal_10seeds/E185_seed1
PASS: recovery run saved.


S2-E15: 1939 spots | K=15 | RNA + ATAC

RECOVERY | PRESENT | S2-E15 | seed=8
Input spots: 1939
Target K   : 15
Loading data and parameters...


Input data has been loaded


Computing METIS partitioning...
Done!
Model training:  42%|████▏     | 42/100 [02:12<03:02,  3.15s/it, NLL_loss=1.04, BNN_loss=0.224, MSE_loss=0.402, IOA_loss=0.0797, ES counter=20, ES patience=20]


Early stop the training process


Succeed to find 15 clusters at resolution 0.750

Target K       : 15
Official output K: 15
Exact K match  : True
ARI = 0.412940880569
NMI = 0.588352603427

Saved: /kaggle/working/PRESENT_baseline/formal_10seeds/S2E15_seed8
PASS: recovery run saved.

RECOVERY COMPLETE
E18.5    seed=1 | target K=14 | actual K=14 | ARI=0.489642 | NMI=0.585966
S2-E15   seed=8 | target K=15 | actual K=15 | ARI=0.412941 | NMI=0.588353


Cell 9.2：重新构建完整 50-run RAW + SUMMARY

In [25]:
# ============================================================
# Cell 9.2
# Rebuild PRESENT formal RAW + SUMMARY
#
# Accept official Leiden fallback outputs when exact target K
# cannot be reached.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


FORMAL_ROOT = Path(
    "/kaggle/working/PRESENT_baseline/formal_10seeds"
)


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


EXPECTED = {

    "HLN-A1": {
        "n_spots": 3484,
        "K": 10,
    },

    "HLN-D1": {
        "n_spots": 3359,
        "K": 11,
    },

    "E18.5": {
        "n_spots": 2129,
        "K": 14,
    },

    "S2-E15": {
        "n_spots": 1939,
        "K": 15,
    },

    "S2-E18": {
        "n_spots": 2248,
        "K": 16,
    },
}


rows = []


print("=" * 110)
print("REBUILD PRESENT FORMAL RESULTS")
print("=" * 110)


for dataset in DATASET_ORDER:

    for seed in range(10):

        run_dir = (
            FORMAL_ROOT
            / (
                dataset_folder_name(
                    dataset
                )
                + f"_seed{seed}"
            )
        )


        metrics_path = (
            run_dir
            / "metrics.json"
        )


        assert metrics_path.exists(), (
            f"Missing metrics: "
            f"{dataset} seed={seed}"
        )


        with metrics_path.open(
            "r",
            encoding="utf-8",
        ) as f:

            m = json.load(f)


        assert (
            m["dataset"]
            == dataset
        )


        assert (
            int(
                m["training_seed"]
            )
            == seed
        )


        assert (
            int(
                m["n_spots"]
            )
            == EXPECTED[
                dataset
            ][
                "n_spots"
            ]
        )


        # Target K must remain frozen.
        assert (
            int(
                m["n_clusters"]
            )
            == EXPECTED[
                dataset
            ][
                "K"
            ]
        )


        # But actual PRESENT Leiden K may differ
        # when its official search cannot hit exact K.
        assert (
            int(
                m[
                    "predicted_clusters"
                ]
            )
            > 1
        )


        assert np.isfinite(
            float(
                m["ARI"]
            )
        )


        assert np.isfinite(
            float(
                m["NMI"]
            )
        )


        rows.append(
            m
        )


raw_df = pd.DataFrame(
    rows
)


assert (
    len(raw_df)
    == 50
)


# ============================================================
# Run counts
# ============================================================

counts = (
    raw_df
    .groupby(
        "dataset"
    )
    .size()
)


for dataset in DATASET_ORDER:

    assert (
        int(
            counts[
                dataset
            ]
        )
        == 10
    )


print(
    "\nRun counts:"
)

print(
    counts
)


# ============================================================
# Save RAW
# ============================================================

RAW_PATH = (
    FORMAL_ROOT
    / "PRESENT_5datasets_10seeds_RAW.csv"
)


raw_df.to_csv(
    RAW_PATH,
    index=False,
)


# ============================================================
# Summary
# ============================================================

summary_rows = []


for dataset in DATASET_ORDER:

    part = (
        raw_df[
            raw_df[
                "dataset"
            ]
            == dataset
        ]
        .sort_values(
            "training_seed"
        )
    )


    assert (
        len(part)
        == 10
    )


    summary_rows.append(
        {

            "dataset":
                dataset,

            "n_runs":
                10,

            "ARI_mean":
                float(
                    part[
                        "ARI"
                    ].mean()
                ),

            "ARI_std":
                float(
                    part[
                        "ARI"
                    ].std(
                        ddof=0
                    )
                ),

            "NMI_mean":
                float(
                    part[
                        "NMI"
                    ].mean()
                ),

            "NMI_std":
                float(
                    part[
                        "NMI"
                    ].std(
                        ddof=0
                    )
                ),

            "exact_K_runs":
                int(
                    (
                        part[
                            "predicted_clusters"
                        ].astype(int)
                        ==
                        EXPECTED[
                            dataset
                        ][
                            "K"
                        ]
                    ).sum()
                ),
        }
    )


summary_df = pd.DataFrame(
    summary_rows
)


SUMMARY_PATH = (
    FORMAL_ROOT
    / "PRESENT_5datasets_10seeds_SUMMARY.csv"
)


summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
)


# ============================================================
# Report
# ============================================================

print(
    "\n" + "=" * 110
)

print(
    "PRESENT FORMAL SUMMARY"
)

print(
    "=" * 110
)


for _, row in (
    summary_df.iterrows()
):

    print(
        f"{row['dataset']:8s} | "
        f"ARI "
        f"{row['ARI_mean']:.6f}"
        f" ± "
        f"{row['ARI_std']:.6f}"
        f" | NMI "
        f"{row['NMI_mean']:.6f}"
        f" ± "
        f"{row['NMI_std']:.6f}"
        f" | exact-K "
        f"{int(row['exact_K_runs'])}/10"
    )


print(
    "\nPASS: 50/50 PRESENT runs assembled."
)

print(
    "PASS: official Leiden fallback retained."
)

print(
    "PASS: std uses ddof=0."
)

REBUILD PRESENT FORMAL RESULTS

Run counts:
dataset
E18.5     10
HLN-A1    10
HLN-D1    10
S2-E15    10
S2-E18    10
dtype: int64

PRESENT FORMAL SUMMARY
HLN-A1   | ARI 0.222241 ± 0.017821 | NMI 0.313690 ± 0.012335 | exact-K 10/10
HLN-D1   | ARI 0.168997 ± 0.017011 | NMI 0.272746 ± 0.015221 | exact-K 10/10
E18.5    | ARI 0.474005 ± 0.040496 | NMI 0.597093 ± 0.012320 | exact-K 10/10
S2-E15   | ARI 0.400654 ± 0.026396 | NMI 0.576737 ± 0.011417 | exact-K 10/10
S2-E18   | ARI 0.359082 ± 0.032965 | NMI 0.491640 ± 0.010674 | exact-K 10/10

PASS: 50/50 PRESENT runs assembled.
PASS: official Leiden fallback retained.
PASS: std uses ddof=0.


Cell 9.3：更新最终 protocol，记录 recovery 情况

In [26]:
# ============================================================
# Cell 9.3
# Finalize PRESENT protocol metadata
#
# Important:
# - 50/50 final runs
# - all final runs have exact target K
# - two runs were re-executed once because the first attempt
#   did not reach exact K and therefore had not been accepted
#   or saved as formal results
# ============================================================

from pathlib import Path
import json


FORMAL_ROOT = Path(
    "/kaggle/working/PRESENT_baseline/formal_10seeds"
)

PROTOCOL_PATH = (
    FORMAL_ROOT
    / "protocol.json"
)


assert PROTOCOL_PATH.exists()


with PROTOCOL_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    protocol = json.load(f)


protocol[
    "final_status"
] = {
    "successful_runs": 50,
    "expected_runs": 50,
    "all_exact_target_K": True,
}


protocol[
    "recovery_note"
] = {
    "reason":
        (
            "Two initial executions completed model training "
            "but PRESENT run_leiden did not reach the requested "
            "cluster count. The wrapper therefore rejected those "
            "executions before accepting them as formal runs."
        ),

    "runs_reexecuted_once": [
        {
            "dataset": "E18.5",
            "seed": 1,
            "initial_actual_K": 15,
            "target_K": 14,
            "final_actual_K": 14,
        },
        {
            "dataset": "S2-E15",
            "seed": 8,
            "initial_actual_K": 14,
            "target_K": 15,
            "final_actual_K": 15,
        },
    ],

    "selection_by_ARI_or_NMI":
        False,

    "final_rule":
        (
            "Final formal results require the official PRESENT "
            "Leiden output to match the benchmark target K."
        ),
}


protocol[
    "clustering"
] = (
    "official PRESENT Leiden; "
    "all 50 final runs reached target K"
)


with PROTOCOL_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        protocol,
        f,
        indent=2,
    )


print("=" * 100)
print("PRESENT FINAL PROTOCOL UPDATE")
print("=" * 100)

print(
    "Protocol:",
    PROTOCOL_PATH
)

print(
    "Successful runs:",
    protocol[
        "final_status"
    ][
        "successful_runs"
    ],
)

print(
    "All exact K:",
    protocol[
        "final_status"
    ][
        "all_exact_target_K"
    ],
)

print(
    "Recovery runs:",
    protocol[
        "recovery_note"
    ][
        "runs_reexecuted_once"
    ],
)

print(
    "\nPASS: final protocol metadata updated."
)

PRESENT FINAL PROTOCOL UPDATE
Protocol: /kaggle/working/PRESENT_baseline/formal_10seeds/protocol.json
Successful runs: 50
All exact K: True
Recovery runs: [{'dataset': 'E18.5', 'seed': 1, 'initial_actual_K': 15, 'target_K': 14, 'final_actual_K': 14}, {'dataset': 'S2-E15', 'seed': 8, 'initial_actual_K': 14, 'target_K': 15, 'final_actual_K': 15}]

PASS: final protocol metadata updated.


Cell 10：50/50 独立审计

In [27]:
# ============================================================
# Cell 10
# PRESENT FINAL independent audit
#
# Independently verify:
#   1. 50/50 runs exist
#   2. seeds 0-9 complete
#   3. frozen spot counts
#   4. embeddings finite
#   5. target K == actual K for all 50
#   6. recomputed ARI/NMI == metrics.json
#   7. recomputed metrics == RAW.csv
#   8. recomputed mean/std == SUMMARY.csv
#   9. std uses ddof=0
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


FORMAL_ROOT = Path(
    "/kaggle/working/PRESENT_baseline/formal_10seeds"
)


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


DATASET_FOLDER = {
    "HLN-A1": "HLNA1",
    "HLN-D1": "HLND1",
    "E18.5": "E185",
    "S2-E15": "S2E15",
    "S2-E18": "S2E18",
}


EXPECTED = {

    "HLN-A1": {
        "n_spots": 3484,
        "K": 10,
    },

    "HLN-D1": {
        "n_spots": 3359,
        "K": 11,
    },

    "E18.5": {
        "n_spots": 2129,
        "K": 14,
    },

    "S2-E15": {
        "n_spots": 1939,
        "K": 15,
    },

    "S2-E18": {
        "n_spots": 2248,
        "K": 16,
    },
}


RAW_PATH = (
    FORMAL_ROOT
    / "PRESENT_5datasets_10seeds_RAW.csv"
)

SUMMARY_PATH = (
    FORMAL_ROOT
    / "PRESENT_5datasets_10seeds_SUMMARY.csv"
)

PROTOCOL_PATH = (
    FORMAL_ROOT
    / "protocol.json"
)


assert RAW_PATH.exists()
assert SUMMARY_PATH.exists()
assert PROTOCOL_PATH.exists()


raw_saved = pd.read_csv(
    RAW_PATH
)

summary_saved = pd.read_csv(
    SUMMARY_PATH
)


audit_rows = []


print("=" * 110)
print("PRESENT FINAL 50-RUN INDEPENDENT AUDIT")
print("=" * 110)


# ============================================================
# 1. Audit each individual run
# ============================================================

for dataset in DATASET_ORDER:

    expected = EXPECTED[
        dataset
    ]


    print(
        f"\nAuditing {dataset}..."
    )


    for seed in range(10):

        run_dir = (
            FORMAL_ROOT
            / (
                DATASET_FOLDER[
                    dataset
                ]
                + f"_seed{seed}"
            )
        )


        required_files = {

            "metrics":
                run_dir
                / "metrics.json",

            "embedding":
                run_dir
                / "embedding.npy",

            "pred":
                run_dir
                / "pred_labels.npy",

            "gt":
                run_dir
                / "gt_labels.npy",

            "coords":
                run_dir
                / "coords.npy",

            "spot_ids":
                run_dir
                / "spot_ids.npy",
        }


        for name, path in (
            required_files.items()
        ):

            assert path.exists(), (
                f"{dataset} seed={seed}: "
                f"missing {name}: {path}"
            )


        # ----------------------------------------------------
        # Load
        # ----------------------------------------------------

        with required_files[
            "metrics"
        ].open(
            "r",
            encoding="utf-8",
        ) as f:

            metrics = json.load(f)


        embedding = np.load(
            required_files[
                "embedding"
            ]
        )

        pred = np.load(
            required_files[
                "pred"
            ],
            allow_pickle=True,
        )

        gt = np.load(
            required_files[
                "gt"
            ],
            allow_pickle=True,
        )

        coords = np.load(
            required_files[
                "coords"
            ]
        )

        spot_ids = np.load(
            required_files[
                "spot_ids"
            ],
            allow_pickle=True,
        )


        # ----------------------------------------------------
        # Identity / population
        # ----------------------------------------------------

        assert (
            metrics["dataset"]
            == dataset
        )

        assert (
            int(
                metrics[
                    "training_seed"
                ]
            )
            == seed
        )

        assert (
            len(gt)
            == expected[
                "n_spots"
            ]
        )

        assert (
            len(pred)
            == expected[
                "n_spots"
            ]
        )

        assert (
            len(spot_ids)
            == expected[
                "n_spots"
            ]
        )

        assert (
            len(
                np.unique(
                    spot_ids
                )
            )
            == expected[
                "n_spots"
            ]
        )


        # ----------------------------------------------------
        # Embeddings / coordinates
        # ----------------------------------------------------

        assert (
            embedding.shape
            == (
                expected[
                    "n_spots"
                ],
                50,
            )
        )

        assert np.isfinite(
            embedding
        ).all()


        assert (
            coords.shape
            == (
                expected[
                    "n_spots"
                ],
                2,
            )
        )

        assert np.isfinite(
            coords
        ).all()


        # ----------------------------------------------------
        # Cluster counts
        # ----------------------------------------------------

        gt_k = len(
            np.unique(gt)
        )

        pred_k = len(
            np.unique(pred)
        )


        assert (
            gt_k
            == expected["K"]
        )


        assert (
            pred_k
            == expected["K"]
        ), (
            f"{dataset} seed={seed}: "
            f"pred K={pred_k}, "
            f"expected={expected['K']}"
        )


        assert (
            int(
                metrics[
                    "n_clusters"
                ]
            )
            == expected["K"]
        )

        assert (
            int(
                metrics[
                    "predicted_clusters"
                ]
            )
            == expected["K"]
        )


        # ----------------------------------------------------
        # Independently recompute metrics
        # ----------------------------------------------------

        ari = adjusted_rand_score(
            gt,
            pred,
        )


        nmi = (
            normalized_mutual_info_score(
                gt,
                pred,
                average_method="max",
            )
        )


        assert np.isclose(
            ari,
            float(
                metrics["ARI"]
            ),
            rtol=0,
            atol=1e-12,
        ), (
            f"{dataset} seed={seed}: "
            "ARI mismatch"
        )


        assert np.isclose(
            nmi,
            float(
                metrics["NMI"]
            ),
            rtol=0,
            atol=1e-12,
        ), (
            f"{dataset} seed={seed}: "
            "NMI mismatch"
        )


        audit_rows.append(
            {

                "dataset":
                    dataset,

                "training_seed":
                    seed,

                "n_spots":
                    expected[
                        "n_spots"
                    ],

                "target_K":
                    expected[
                        "K"
                    ],

                "predicted_K":
                    pred_k,

                "ARI":
                    float(
                        ari
                    ),

                "NMI":
                    float(
                        nmi
                    ),
            }
        )


    print(
        f"{dataset}: 10/10 PASS"
    )


# ============================================================
# 2. Build independent RAW
# ============================================================

audit_raw = pd.DataFrame(
    audit_rows
)


assert (
    len(audit_raw)
    == 50
)


# Ensure every dataset has exactly seeds 0-9.
for dataset in DATASET_ORDER:

    part = audit_raw[
        audit_raw[
            "dataset"
        ]
        == dataset
    ]


    assert (
        len(part)
        == 10
    )


    assert (
        sorted(
            part[
                "training_seed"
            ].tolist()
        )
        == list(
            range(10)
        )
    )


# ============================================================
# 3. Compare against saved RAW.csv
# ============================================================

left = (
    audit_raw[
        [
            "dataset",
            "training_seed",
            "ARI",
            "NMI",
        ]
    ]
    .sort_values(
        [
            "dataset",
            "training_seed",
        ]
    )
    .reset_index(
        drop=True
    )
)


right = (
    raw_saved[
        [
            "dataset",
            "training_seed",
            "ARI",
            "NMI",
        ]
    ]
    .sort_values(
        [
            "dataset",
            "training_seed",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(right)
    == 50
)


assert (
    left[
        [
            "dataset",
            "training_seed",
        ]
    ]
    .equals(
        right[
            [
                "dataset",
                "training_seed",
            ]
        ]
    )
)


assert np.allclose(
    left["ARI"],
    right["ARI"],
    rtol=0,
    atol=1e-12,
)


assert np.allclose(
    left["NMI"],
    right["NMI"],
    rtol=0,
    atol=1e-12,
)


print(
    "\nPASS: RAW.csv verified."
)


# ============================================================
# 4. Independently rebuild summary
# ============================================================

audit_summary_rows = []


for dataset in DATASET_ORDER:

    part = audit_raw[
        audit_raw[
            "dataset"
        ]
        == dataset
    ]


    audit_summary_rows.append(
        {

            "dataset":
                dataset,

            "n_runs":
                10,

            "ARI_mean":
                float(
                    part[
                        "ARI"
                    ].mean()
                ),

            "ARI_std":
                float(
                    part[
                        "ARI"
                    ].std(
                        ddof=0
                    )
                ),

            "NMI_mean":
                float(
                    part[
                        "NMI"
                    ].mean()
                ),

            "NMI_std":
                float(
                    part[
                        "NMI"
                    ].std(
                        ddof=0
                    )
                ),

            "exact_K_runs":
                int(
                    (
                        part[
                            "predicted_K"
                        ]
                        ==
                        part[
                            "target_K"
                        ]
                    ).sum()
                ),
        }
    )


audit_summary = pd.DataFrame(
    audit_summary_rows
)


# ============================================================
# 5. Compare against saved SUMMARY.csv
# ============================================================

summary_right = (
    summary_saved
    .set_index(
        "dataset"
    )
    .loc[
        DATASET_ORDER
    ]
    .reset_index()
)


for col in [
    "ARI_mean",
    "ARI_std",
    "NMI_mean",
    "NMI_std",
]:

    assert np.allclose(
        audit_summary[
            col
        ],
        summary_right[
            col
        ],
        rtol=0,
        atol=1e-12,
    ), (
        f"SUMMARY mismatch: {col}"
    )


assert (
    audit_summary[
        "exact_K_runs"
    ]
    == 10
).all()


if (
    "exact_K_runs"
    in summary_right.columns
):

    assert (
        summary_right[
            "exact_K_runs"
        ].astype(int)
        == 10
    ).all()


print(
    "PASS: SUMMARY.csv verified."
)


# ============================================================
# 6. Protocol audit
# ============================================================

with PROTOCOL_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    protocol = json.load(f)


assert (
    protocol[
        "final_status"
    ][
        "successful_runs"
    ]
    == 50
)

assert (
    protocol[
        "final_status"
    ][
        "all_exact_target_K"
    ]
    is True
)


print(
    "PASS: protocol.json verified."
)


# ============================================================
# 7. Save independent audit
# ============================================================

AUDIT_RAW_PATH = (
    FORMAL_ROOT
    / "PRESENT_5datasets_10seeds_AUDIT_RAW.csv"
)


AUDIT_SUMMARY_PATH = (
    FORMAL_ROOT
    / "PRESENT_5datasets_10seeds_AUDIT_SUMMARY.csv"
)


audit_raw.to_csv(
    AUDIT_RAW_PATH,
    index=False,
)


audit_summary.to_csv(
    AUDIT_SUMMARY_PATH,
    index=False,
)


# ============================================================
# 8. Final report
# ============================================================

print(
    "\n" + "=" * 110
)

print(
    "PRESENT INDEPENDENT SUMMARY"
)

print(
    "=" * 110
)


for _, row in (
    audit_summary.iterrows()
):

    print(
        f"{row['dataset']:8s} | "
        f"ARI "
        f"{row['ARI_mean']:.6f}"
        f" ± "
        f"{row['ARI_std']:.6f}"
        f" | NMI "
        f"{row['NMI_mean']:.6f}"
        f" ± "
        f"{row['NMI_std']:.6f}"
        f" | exact-K "
        f"{int(row['exact_K_runs'])}/10"
    )


print(
    "\nPASS: 50/50 runs independently verified."
)

print(
    "PASS: all 50 predictions have exact target K."
)

print(
    "PASS: metrics.json == recomputed metrics."
)

print(
    "PASS: RAW.csv == independent recomputation."
)

print(
    "PASS: SUMMARY.csv == independent recomputation."
)

print(
    "PASS: std uses ddof=0."
)

print(
    "PASS: frozen benchmark populations verified."
)

PRESENT FINAL 50-RUN INDEPENDENT AUDIT

Auditing HLN-A1...
HLN-A1: 10/10 PASS

Auditing HLN-D1...
HLN-D1: 10/10 PASS

Auditing E18.5...
E18.5: 10/10 PASS

Auditing S2-E15...
S2-E15: 10/10 PASS

Auditing S2-E18...
S2-E18: 10/10 PASS

PASS: RAW.csv verified.
PASS: SUMMARY.csv verified.
PASS: protocol.json verified.

PRESENT INDEPENDENT SUMMARY
HLN-A1   | ARI 0.222241 ± 0.017821 | NMI 0.313690 ± 0.012335 | exact-K 10/10
HLN-D1   | ARI 0.168997 ± 0.017011 | NMI 0.272746 ± 0.015221 | exact-K 10/10
E18.5    | ARI 0.474005 ± 0.040496 | NMI 0.597093 ± 0.012320 | exact-K 10/10
S2-E15   | ARI 0.400654 ± 0.026396 | NMI 0.576737 ± 0.011417 | exact-K 10/10
S2-E18   | ARI 0.359082 ± 0.032965 | NMI 0.491640 ± 0.010674 | exact-K 10/10

PASS: 50/50 runs independently verified.
PASS: all 50 predictions have exact target K.
PASS: metrics.json == recomputed metrics.
PASS: RAW.csv == independent recomputation.
PASS: SUMMARY.csv == independent recomputation.
PASS: std uses ddof=0.
PASS: frozen benchmark pop

Cell 11：最终归档 ZIP

In [28]:
# ============================================================
# Cell 11
# PRESENT FINAL archive
# ============================================================

from pathlib import Path
import shutil
import hashlib


FORMAL_ROOT = Path(
    "/kaggle/working/PRESENT_baseline/formal_10seeds"
)


ARCHIVE_NAME = (
    "PRESENT_5datasets_10seeds_FORMAL_FINAL"
)


STAGE_DIR = (
    Path("/kaggle/working")
    / ARCHIVE_NAME
)


ZIP_BASE = (
    Path("/kaggle/working")
    / ARCHIVE_NAME
)


# ============================================================
# 1. Reset staging area
# ============================================================

if STAGE_DIR.exists():

    shutil.rmtree(
        STAGE_DIR
    )


ZIP_PATH = Path(
    str(ZIP_BASE)
    + ".zip"
)


if ZIP_PATH.exists():

    ZIP_PATH.unlink()


STAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Copy top-level evidence
# ============================================================

TOP_FILES = [

    "protocol.json",

    "PRESENT_5datasets_10seeds_RAW.csv",

    "PRESENT_5datasets_10seeds_SUMMARY.csv",

    "PRESENT_5datasets_10seeds_AUDIT_RAW.csv",

    "PRESENT_5datasets_10seeds_AUDIT_SUMMARY.csv",
]


for name in TOP_FILES:

    src = (
        FORMAL_ROOT
        / name
    )


    assert src.exists(), src


    shutil.copy2(
        src,
        STAGE_DIR
        / name,
    )


# ============================================================
# 3. Copy all 50 formal runs
# ============================================================

DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


DATASET_FOLDER = {
    "HLN-A1": "HLNA1",
    "HLN-D1": "HLND1",
    "E18.5": "E185",
    "S2-E15": "S2E15",
    "S2-E18": "S2E18",
}


RUN_FILES = [

    "metrics.json",

    "embedding.npy",

    "pred_labels.npy",

    "gt_labels.npy",

    "coords.npy",

    "spot_ids.npy",
]


for dataset in DATASET_ORDER:

    dataset_dst = (
        STAGE_DIR
        / DATASET_FOLDER[
            dataset
        ]
    )


    dataset_dst.mkdir(
        parents=True,
        exist_ok=True,
    )


    for seed in range(10):

        src_dir = (
            FORMAL_ROOT
            / (
                DATASET_FOLDER[
                    dataset
                ]
                + f"_seed{seed}"
            )
        )


        dst_dir = (
            dataset_dst
            / f"seed{seed}"
        )


        dst_dir.mkdir(
            parents=True,
            exist_ok=True,
        )


        for name in RUN_FILES:

            src = (
                src_dir
                / name
            )


            assert src.exists(), (
                f"Missing: {src}"
            )


            shutil.copy2(
                src,
                dst_dir
                / name,
            )


# ============================================================
# 4. Archive README
# ============================================================

README = """PRESENT formal baseline archive

Experiment
==========
Datasets:
- HLN-A1
- HLN-D1
- E18.5
- S2-E15
- S2-E18

Runs:
- seeds 0 through 9
- 10 runs per dataset
- 50 formal runs total

Final formal results:
- all 50 runs completed
- all 50 official PRESENT Leiden outputs reached target K

PRESENT configuration
=====================
gene_min_cells = 1
num_hvg = 3000
protein_min_cells = 1 for RNA+ADT
peak_min_cells_fraction = 0.03 for RNA+ATAC

d_lat = 50
k_neighbors = 6
max epochs = 100
learning rate = 0.001
batch size = 320

Official PRESENT early stopping retained.
Official PRESENT Leiden clustering retained.

Evaluation
==========
ARI:
sklearn.metrics.adjusted_rand_score

NMI:
sklearn.metrics.normalized_mutual_info_score
average_method = "max"

Standard deviation:
ddof = 0

Benchmark-population compatibility
==================================
Observation-level filtering inside PRESENT was disabled
to preserve identical benchmark spot populations across
all compared methods.

Feature-level preprocessing was retained.

HLN-A1 note
============
PRESENT's observation filtering would remove one ADT
zero-count spot:

GGACGTCGCACGAGAA-1

This spot was retained so that HLN-A1 remains the common
3484-spot benchmark population.

Compatibility
=============
epiScanpy = 0.3.2

NumPy 2.x legacy aliases were restored where required.
SciPy sparse .A compatibility was restored.

No PRESENT model architecture, objective, training rule,
or clustering algorithm was replaced.

Recovery audit
==============
Two first executions:
- E18.5 seed 1
- S2-E15 seed 8

completed model training but did not reach the requested
cluster count during PRESENT's Leiden resolution search.
The wrapper therefore did not accept those executions as
formal results.

Each was re-executed once with the same frozen protocol
and seed. Both re-executions reached the exact target K.

No ARI/NMI-based model or run selection was performed.

See protocol.json for the machine-readable audit record.
"""


(
    STAGE_DIR
    / "README_ARCHIVE.txt"
).write_text(
    README,
    encoding="utf-8",
)


# ============================================================
# 5. SHA256 checksums
# ============================================================

def sha256_file(
    path,
):

    hasher = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )


            if not block:
                break


            hasher.update(
                block
            )


    return hasher.hexdigest()


manifest_lines = []


for path in sorted(
    STAGE_DIR.rglob("*")
):

    if not path.is_file():
        continue


    if (
        path.name
        == "SHA256SUMS.txt"
    ):
        continue


    rel = path.relative_to(
        STAGE_DIR
    )


    digest = sha256_file(
        path
    )


    manifest_lines.append(
        f"{digest}  {rel}"
    )


(
    STAGE_DIR
    / "SHA256SUMS.txt"
).write_text(
    "\n".join(
        manifest_lines
    )
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 6. Structural archive audit
# ============================================================

assert (
    len(
        list(
            STAGE_DIR.rglob(
                "metrics.json"
            )
        )
    )
    == 50
)


assert (
    len(
        list(
            STAGE_DIR.rglob(
                "embedding.npy"
            )
        )
    )
    == 50
)


assert (
    len(
        list(
            STAGE_DIR.rglob(
                "pred_labels.npy"
            )
        )
    )
    == 50
)


assert (
    len(
        list(
            STAGE_DIR.rglob(
                "gt_labels.npy"
            )
        )
    )
    == 50
)


assert (
    len(
        list(
            STAGE_DIR.rglob(
                "spot_ids.npy"
            )
        )
    )
    == 50
)


# ============================================================
# 7. Zip
# ============================================================

zip_path = shutil.make_archive(
    str(
        ZIP_BASE
    ),
    "zip",
    root_dir=STAGE_DIR.parent,
    base_dir=STAGE_DIR.name,
)


zip_path = Path(
    zip_path
)


print("=" * 110)
print("PRESENT FINAL ARCHIVE")
print("=" * 110)

print(
    "Formal runs    : 50"
)

print(
    "metrics.json   :",
    len(
        list(
            STAGE_DIR.rglob(
                "metrics.json"
            )
        )
    )
)

print(
    "embeddings     :",
    len(
        list(
            STAGE_DIR.rglob(
                "embedding.npy"
            )
        )
    )
)

print(
    "SHA256 entries :",
    len(
        manifest_lines
    )
)

print(
    "\nZIP:",
    zip_path
)

print(
    "ZIP size:",
    f"{zip_path.stat().st_size / 1024 / 1024:.2f} MB"
)

print(
    "\nPASS: PRESENT FINAL archive created."
)

PRESENT FINAL ARCHIVE
Formal runs    : 50
metrics.json   : 50
embeddings     : 50
SHA256 entries : 306

ZIP: /kaggle/working/PRESENT_5datasets_10seeds_FORMAL_FINAL.zip
ZIP size: 24.44 MB

PASS: PRESENT FINAL archive created.
